The first functions are to be applied on symbolic expressions (By+iBx from bpmeth expansion)

In [ ]:
def rt2xy(r,theta):
    return r*np.cos(theta),r*np.sin(theta) #cartesian positions of points on which i have the field B_func 
sindex=25
s_fixed=ss[sindex]
nk=4 #maximum abs of k
nl=nk//2+1

def ByiBx(x, y):#analytical definition of By + i*Bx at s=s_fixed, needs ndarrays comng from bpmeth field expansion
    return By_func(x, y, s_fixed) + 1j * Bx_func(x, y, s_fixed)
rr=np.linspace(rmin,rmax,nr)

def dkharmonics(ByiBx,rr=rr,nk=nk,ntheta=102):
  '''
  returns dk as function of r 
  ''' 
   #radial positions
  out=np.empty((len(rr),2*nk+1),dtype=complex) #prepare to store dk's as functions of r0 
  theta=np.arange(ntheta)/ntheta*2*np.pi
  for ir,r in enumerate(rr):
    x,y=rt2xy(r,theta)
    b=ByiBx(x,y)
    d=np.fft.fft(b)/ntheta #discrete fourier coefficients: dk at this r0
    out[ir,0]=d[0]#k=0
    out[ir,1::2]=d[1:nk+1] #first "fourier frequencies" are for positive k, stored in odd indexes
    out[ir,2::2]=d[ntheta:ntheta-nk-1:-1]#the last ones are for negative frequencies (take them in reverse order), stored in even indexes
  return out

def dklfitraw(dkofrarray, rr, nk, nl):
    '''
    takes the result from dkharmonics and fits d_k/r^|k| for different k values as a function of r^2l,
    the coefficients of the fit are the values d_kl
    parameters:
      rr    = array of r values
      nk    = max of |k| (not the number of different k values)
      nl    = number of l values
    returns: matrix dkl
    '''
    polynomial = np.zeros((len(rr), 2*nk+1), dtype=complex)
    ka = np.linspace(-nk, nk, 2*nk+1)

    for ir, r in enumerate(rr):
        polynomial[ir, 0]    = dkofrarray[ir, 0]
        polynomial[ir, 1::2] = dkofrarray[ir, 1::2] / r**ka[nk+1 : 2*nk+1]
        polynomial[ir, 2::2] = dkofrarray[ir, 2::2] / r**-ka[0: nk]

    # Build Vandermonde matrix for r^2, degrees 0..nl-1
    x = rr**2
    V = np.vander(x, nl, increasing=True)  # shape (len(rr), nl)

    # Fit all 2*nk+1 columns at once: coeffs shape is (nl, 2*nk+1)
    coeffs, _, _, _ = np.linalg.lstsq(V, polynomial, rcond=None)

    # dkl[k, l] layout: rows = k index, cols = l index
    dkl = coeffs.T  # shape (2*nk+1, nl)

    return dkl

#now let's recover the a,b coefficients
def bnianHAopt(dkl, nl, nk):
    # Precompute factorials
    fact = np.array([factorial(n) for n in range(nk)], dtype=np.int16)

    # Build k index dict mapping: row index for k = 0,1,-1,2,-2,...
    k_index = {0: 0}
    for i in range(1, nk+1):
        k_index[i]  = 2*i - 1
        k_index[-i] = 2*i #negative k gets even row

    result = np.zeros(nk, dtype=np.complex128)

    for n in range(nk):
        s = 0.0 + 0.0j #initialize sum
        max_l = min(n//2, nl-1)

        for l in range(max_l + 1):
            k = n - 2*l
            if k == 0:
                s += dkl[k_index[0], l]
            else:
                s += dkl[k_index[k], l] + dkl[k_index[-k], l]

        result[n] = fact[n] * s

    return result
def mylocalharmonics():
    '''
    local harmonic analysis from By+iBx to dkl and coefficients a,b 
    '''
    dk=dkharmonics(ByiBx=ByiBx, rr=rr,nk=nk,ntheta=102)
    dkl=dklfitraw(dk,rr,nk, nl)
    an=bnianHAopt(dkl, nl,nk).imag
    bn=bnianHAopt(dkl, nl,nk).real
    return an, bn

In [ ]:
#timing
def mylocalharmonics():
    '''
    local harmonic analysis from By+iBx to dkl and coefficients a,b 
    '''
    dk=dkharmonics(ByiBx=ByiBx, rr=rr,nk=nk,ntheta=102)
    dkl=dklfit(dk,rr,nk, nl)
    an=bnianHAopt(dkl, nl,nk).imag
    bn=bnianHAopt(dkl, nl,nk).real
    return an, bn

def bpmethlocalharmonics():
    dkl=bpmeth.harmonics(ByiBx=ByiBx,nk=nk,rmin=rmin, rmax=rmax,nr=nr, ntheta=102)
    an=bpmeth.calc_coeffs(dkl).imag
    bn=bpmeth.calc_coeffs(dkl).real
    return an, bn
N=1000
import timeit
mytime=timeit.timeit("mylocalharmonics()", globals=globals(), number=N)
bpmethtime=timeit.timeit("bpmethlocalharmonics()", globals=globals(), number=N)
print("my time=", mytime)
print("bpmeth time", bpmethtime)
print("relative improvement timed over", N,"repetitions: (bpmeth time - new code time)/bpmeth time =", (bpmethtime-mytime)/bpmethtime*100,"%")

my time= 22.60738687400044
bpmeth time 24.8815770709989
relative improvement timed over 1000 repetitions: (bpmeth time - new code time)/bpmeth time = 9.140056478370004 %


Chebyschev polynomials instead of Vandermonde.

In [ ]:
#this is a bit slower because it deals with real and imag part separately in loop. better to use vandermonde as in dklfitraw
from numpy.polynomial.chebyshev import chebvander, cheb2poly

def dklfit_cheb(dkofrarray, rr, nk, nl):
    '''
    Same as dklfit but fits in the Chebyshev basis for better numerical stability.
    The fit variable is x = r^2, mapped to [-1, 1] for the Chebyshev domain.
    
    Returns: dkl in the MONOMIAL basis (same layout as dklfit),
             converted from Chebyshev coefficients via cheb2poly.
    '''
    polynomial = np.zeros((len(rr), 2*nk+1), dtype=complex)
    ka = np.linspace(-nk, nk, 2*nk+1)

    for ir, r in enumerate(rr):
        polynomial[ir, 0]    = dkofrarray[ir, 0]
        polynomial[ir, 1::2] = dkofrarray[ir, 1::2] / r**ka[nk+1 : 2*nk+1]
        polynomial[ir, 2::2] = dkofrarray[ir, 2::2] / r**-ka[0: nk]

    # Map r^2 to [-1, 1] for Chebyshev domain
    x = rr**2
    x_min, x_max = x.min(), x.max()
    x_scaled = 2 * (x - x_min) / (x_max - x_min) - 1  # in [-1, 1]

    # Chebyshev Vandermonde matrix: shape (len(rr), nl)
    V = chebvander(x_scaled, nl - 1)

    # Fit all k-columns at once
    cheb_coeffs, _, _, _ = np.linalg.lstsq(V, polynomial, rcond=None)
    # cheb_coeffs shape: (nl, 2*nk+1)

    # Convert each k column from Chebyshev to monomial basis
    # cheb2poly works on real arrays, so handle real/imag separately
    dkl = np.zeros((2*nk+1, nl), dtype=complex)
    for ik in range(2*nk+1):
        mono_re = cheb2poly(cheb_coeffs[:, ik].real)
        mono_im = cheb2poly(cheb_coeffs[:, ik].imag)
        dkl[ik, :] = mono_re + 1j * mono_im

    return dkl

To be used on numerical arrays:

In [ ]:
def dkongrid(Byibx_rts, rr, nk, s_index, ntheta=102):
    """
    Same as dkharmonics but takes a precomputed field array
    instead of a callable.

    Parameters
    ----------
    Byibxrts : complex array, shape (nr, ntheta, ns)
        (By + i*Bx) already evaluated on the (r, theta, s) grid.
        Theta samples must be uniform in [0, 2*pi).
    rr     : array of r values, shape (nr,)
    nk     : max |k|
    s_index: index along the s-axis to use

    Returns
    -------
    out : complex array, shape (nr, 2*nk+1)  — same as dkharmonics
    """
    ntheta = Byibx_rts.shape[1]

    b = Byibx_rts[:, :, s_index]        # (nr, ntheta)
    d = np.fft.fft(b, axis=1) / ntheta       # FFT along theta axis

    out = np.empty((len(rr), 2*nk+1), dtype=complex)
    out[:, 0]    = d[:, 0]
    out[:, 1::2] = d[:, 1 : nk+1]
    out[:, 2::2] = d[:, ntheta : ntheta-nk-1 : -1]

    return out
def dklfitrawcut(dkofrarray, rr, nk, nl, rmin_fit=None):
    '''
    rmin_fit: minimum r to include in the fit (to avoid blow-up of dk/r^|k| at small r).
              Defaults to rr.max() * 0.05 if not given.
    '''
    if rmin_fit is None:
        rmin_fit = rr.max() * 0.6

    mask = rr >= rmin_fit
    rr_fit = rr[mask]
    dkofr_fit = dkofrarray[mask, :]

    polynomial = np.zeros((len(rr_fit), 2*nk+1), dtype=complex)
    kpos = np.arange(1, nk+1)   # [1, 2, ..., nk]

    for ir, r in enumerate(rr_fit):
        polynomial[ir, 0]    = dkofr_fit[ir, 0]
        polynomial[ir, 1::2] = dkofr_fit[ir, 1::2] / r**kpos   # positive k
        polynomial[ir, 2::2] = dkofr_fit[ir, 2::2] / r**kpos   # negative k, same |k|

    x = rr_fit**2
    V = np.vander(x, nl, increasing=True)
    coeffs, _, _, _ = np.linalg.lstsq(V, polynomial, rcond=None)

    return coeffs.T   # shape (2*nk+1, nl)
def LHongrid(ByiBxongrid, s_index):
    '''
    local harmonic analysis from By+iBx to dkl and coefficients a,b 
    '''
    dk=dkongrid(Byibx_rts=ByiBxongrid, rr=rr, nk=nk, s_index=s_index)
    dkl=dklfitrawcut(dk,rr,nk, nl)
    an=bnianHAopt(dkl, nl,nk).imag
    bn=bnianHAopt(dkl, nl,nk).real
    return an, bn